# Week 3, Lab 3 — Sequential vs hierarchical


In [1]:
WEEK = 'Week 3'
LAB = 'Lab 3 — process types'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 3 / Lab 3 — process types
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
from crewai import LLM, Agent, Task, Crew, Process

# cfg
llm = LLM(
    model="ollama/qwen2.5:3b",
    base_url="http://localhost:11434/",
)
print("CrewAI LLM ->", cfg)

CrewAI LLM -> {'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'qwen2.5:3b'}


In [3]:
researcher = Agent(role="Researcher", goal="Collect 3 facts.", backstory="Analyst.", llm=llm)
writer = Agent(role="Writer", goal="Write a short paragraph.", backstory="Teacher.", llm=llm)
manager = Agent(role="Manager", goal="Delegate and deliver a clean student paragraph.", backstory="Project lead.", llm=llm)

t_research = Task(description="3 facts about Ollama.", expected_output="3 bullets", agent=researcher)
t_write = Task(
    description="One short paragraph for students using those facts.",
    expected_output="1 paragraph",
    context=[t_research],
    agent=writer,
)

print("==== SEQUENTIAL ====")
print(Crew(agents=[researcher, writer], tasks=[t_research, t_write], process=Process.sequential).kickoff())

print("\n==== HIERARCHICAL ====")
print(Crew(
    agents=[researcher, writer, manager],
    tasks=[t_research, t_write],
    process=Process.hierarchical,
    manager_llm=llm,
).kickoff())

==== SEQUENTIAL ====
OpenAI's Ollama platform, formerly known as DALL-E 2, has revolutionized image generation by allowing users to create visual content based on textual descriptions. Powered by a vast library of over 5 billion textual prompts derived from an extensive dataset that includes diverse sources such as news articles, literature, forums, social media posts, and more, Ollama demonstrates its capability to produce images with high accuracy and creativity. The platform's unique strength lies in its training on text spanning multiple languages including English, Chinese, Spanish, French, German, Italian, Portuguese, Japanese, Korean, among others, making it a versatile model capable of understanding and generating responses across various linguistic contexts, thus enabling users to explore and connect with global cultures and ideas more effectively.

==== HIERARCHICAL ====
Ollama, a concept found in science fiction and fantasy literature, embodies an array of extraordinary phys

Hierarchical uses more tokens and can flake on tiny models — that is a reliability lesson. **Next:** tools.
